# Iteration 2 – Reporting propensity by Council District (Shamie)

Task file: `docs/tasks/iter2-shamie.md` · Reviewer: Ma'el · Outputs: `outputs/iter2/propensity/`

**Question (client Q1):** are GlendaleOne request counts in line with each Council District's population, and which district characteristics line up with the request rate?

**Inputs:** `data/cleaned/glendaleone_clean.csv` (GlendaleOne External Requests, extract loaded 2026-08-06, team Drive copy), `data/cleaned/acs_demo_income_profile_clean.csv` (Esri 2026 Demographic and Income Profile) and `data/cleaned/acs_population_summary_clean.csv` (ACS 2020-24 5-year Population Summary).

**Grain:** one row = one GlendaleOne request. The two demographic files have one row per district.

**Rate:** requests per 1,000 residents per year = count / (79 / 12) / population x 1,000. Esri 2026 population (`dip_summary_total_population_2026`) is the primary denominator; ACS 2020-24 (`acs_total_population`) is the sensitivity check.

**Wording rule:** associative only ("lines up with", "is associated with", "co-varies with"). With six districts every correlation is descriptive.

Run with **Kernel → Restart & Run All**, or from the repo root:
`jupyter nbconvert --to notebook --execute --inplace notebooks/iter2_propensity_shamie.ipynb`.
Clear outputs before committing: `jupyter nbconvert --clear-output --inplace notebooks/iter2_propensity_shamie.ipynb`.

## 0. Setup

Standard load block, verbatim from `docs/ANALYSIS_STANDARDS.md` section 2.

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data" / "cleaned"
DISTRICTS = ["BARREL", "CACTUS", "CHOLLA", "OCOTILLO", "SAHUARO", "YUCCA"]
START, END = "2020-01-01", "2026-07-31"      # 79 full months
YEARS = 79 / 12

req = pd.read_csv(DATA / "glendaleone_clean.csv",
                  parse_dates=["request_date", "close_date"], low_memory=False)
req["district"] = req["district"].astype("string").str.strip().str.upper()
n_raw = len(req)
req_w = req[req["request_date"].between(START, END)]
n_window = len(req_w)
req_w = req_w[req_w["district"].isin(DISTRICTS)].copy()
n_final = len(req_w)
print(n_raw, n_window, n_final, n_window - n_final)   # last number = rows without a district

Chart standard block, verbatim from `docs/ANALYSIS_STANDARDS.md` section 7, then the other imports, the output folder and shared helpers.

- `nice()` turns `BARREL` into `Barrel` for anything a person reads (chart labels and titles). The `district` column in every CSV stays UPPERCASE: it is the join key and matches the time-to-close tables.
- `rate_per_1000()` is the rate formula from section 3: count / YEARS / population x 1,000.
- `fmt_p()` writes p as `< .001` below .001 (section 6).

In [ ]:
import seaborn as sns, matplotlib.pyplot as plt
sns.set_theme(style="whitegrid", context="talk")        # seaborn 0.13.2
BLUE = "#2a78d6"                                        # default single hue
DISTRICT_COLORS = {"BARREL": "#2a78d6", "CACTUS": "#eb6834", "CHOLLA": "#1baf7a",
                   "OCOTILLO": "#eda100", "SAHUARO": "#e87ba4", "YUCCA": "#008300"}

In [ ]:
import textwrap
import numpy as np
from scipy import stats
from matplotlib.lines import Line2D
from matplotlib.ticker import FuncFormatter

OUT = ROOT / "outputs" / "iter2" / "propensity"
OUT.mkdir(parents=True, exist_ok=True)

TRASH = "Trash/Recycle Services"
POP_ESRI = "dip_summary_total_population_2026"
POP_ACS = "acs_total_population"
SOURCE = ("Source: City of Glendale GlendaleOne External Requests (extract loaded 2026-08-06); "
          "Esri 2026 population.")
WINDOW_TXT = "2020-01 to 2026-07 (79 months)"
GREY = "#4d4d4d"
LINE_GREY = "#c5ccd3"


def nice(d):
    return str(d).title()


def fmt_p(p):
    return "< .001" if p < 0.001 else f"{p:.3f}".lstrip("0")


def rate_per_1000(count, population):
    return count / YEARS / population * 1000

## 1. Data checks

The checks from `docs/ANALYSIS_STANDARDS.md` section 5 that apply to counting requests by district: the filter log, dtypes, missing values in the columns used (NULL, `''`, `'N/A'` and `'NONE'` all count as missing), unique `request_number`, dates inside the window, exactly the six district names, no empty months, and the per-district counts adding up to `n_final`. The mean, median and outlier checks do not apply here: this analysis counts requests and measures no continuous value per request.

In [ ]:
n_before = int((req["request_date"] < START).sum())
n_after = int((req["request_date"] > END).sum())
n_no_date = int(req["request_date"].isna().sum())
assert n_before + n_after + n_no_date == n_raw - n_window
print({"n_raw": n_raw, "before_window": n_before, "after_window": n_after, "no_request_date": n_no_date,
       "n_window": n_window, "excluded_no_district": n_window - n_final, "n_final": n_final,
       "excluded_share_of_window_pct": round((n_window - n_final) / n_window * 100, 2)})

assert pd.api.types.is_datetime64_any_dtype(req_w["request_date"])
assert pd.api.types.is_string_dtype(req_w["district"])
assert req_w["request_number"].is_unique
assert req_w["request_date"].min() >= pd.Timestamp(START)
assert req_w["request_date"].max() <= pd.Timestamp(END)
assert set(req_w["district"].unique()) == set(DISTRICTS)

used = ["request_number", "request_date", "district", "request_type_group"]
missing = pd.DataFrame({c: req_w[c].isna() | req_w[c].astype("string").str.strip().isin(["", "N/A", "NONE"])
                        for c in used})
print(missing.sum().to_dict())
assert missing.sum().sum() == 0

months = req_w["request_date"].dt.to_period("M").value_counts()
full = pd.period_range(START, END, freq="M")
assert len(full) == 79 and months.reindex(full, fill_value=0).gt(0).all()
print(f"{len(full)} months in the window, none empty; rows per month from {months.min():,} to {months.max():,}")

In [ ]:
dip = pd.read_csv(DATA / "acs_demo_income_profile_clean.csv")
acs = pd.read_csv(DATA / "acs_population_summary_clean.csv")

for name, df in {"dip": dip, "acs": acs}.items():
    df["district"] = df["district"].astype("string").str.strip().str.upper()
    assert len(df) == 6 and df["district"].is_unique
    assert set(df["district"]) == set(DISTRICTS)
    assert not df.drop(columns="district").isna().any().any()
    pct_cols = [c for c in df.columns if c.endswith("_pct")]
    assert ((df[pct_cols] >= 0) & (df[pct_cols] <= 100)).all().all()
    print(name, df.shape)

demo = (dip.merge(acs, on="district", validate="one_to_one")
           .set_index("district").loc[DISTRICTS].reset_index())

households = (demo["dip_summary_owner_occupied_housing_units_2026"]
              + demo["dip_summary_renter_occupied_housing_units_2026"])
assert households.eq(demo["dip_summary_total_households_2026"]).all()

rb_all = [c for c in acs.columns if c.startswith("acs_rent_burden_") and c.endswith("_pct")]
print("ACS rent-burden bands sum to", demo[rb_all].sum(axis=1).round(1).tolist())

pd.DataFrame({"district": demo["district"],
              "esri_population_2026": demo[POP_ESRI],
              "acs_population_2020_24": demo[POP_ACS],
              "esri_vs_acs_pct": ((demo[POP_ESRI] / demo[POP_ACS] - 1) * 100).round(1)})

## 2. Planned tests (written before running them)

The same three tests as the first version of this notebook, with the four scenarios written out, plus the QA two-way check (`docs/ANALYSIS_STANDARDS.md` sections 6 and 8).

1. **Chi-square goodness of fit** (`scipy.stats.chisquare(obs, f_exp=exp)`): district request counts against the counts expected from each district's population share. Cohen's w = sqrt(chi2 / N) leads, labelled with Cohen's (1988) rule of thumb (under 0.1 negligible, 0.1 small, 0.3 medium, 0.5 large); p is reported as `< .001` below .001. Per district: the standardized residual (obs - exp) / sqrt(exp) and the percent above or below expected. Four scenarios: all requests and Trash/Recycle Services removed, each with the Esri 2026 and the ACS 2020-24 population. The per-district diagnostics are saved for the primary scenario (all requests, Esri).
2. **Spearman rho** (`scipy.stats.spearmanr`) between the district request rate and five covariates: median household income (`dip_hhinc_median_household_income_2026`), renter share (Esri 2026 renter-occupied units / households), households with no internet access (`acs_internet_with_no_internet_access_pct`), adults 18-64 who speak Spanish and speak English "not well" (`acs_lang_18_64_speak_spanish_speak_english_not_well_pct`) and renter households paying 30% or more of income on rent. n = 6, so rho is descriptive and no p-value is reported. Run for the same four rate versions.
3. **Sensitivity:** does the district ranking change with the ACS population, and with Trash/Recycle Services removed? Kept as rank columns (1 = highest rate) in `request_rates.csv`.
4. **Two-way check:** per-district counts by `groupby` and by `value_counts` must match, and chi-square recomputed by hand must match scipy.

### 2a. Request rates by district → `request_rates.csv`

Trash/Recycle Services is matched on the trimmed `request_type_group` (`docs/PROJECT.md` 6.12). Rates are kept unrounded in memory and rounded to one decimal only when written; ranks are taken from the unrounded rates.

In [ ]:
is_trash = req_w["request_type_group"].astype("string").str.strip().eq(TRASH)
n_trash = int(is_trash.sum())
req_nt = req_w[~is_trash]
n_no_trash = len(req_nt)

counts = req_w.groupby("district").size().reindex(DISTRICTS)
counts_vc = req_w["district"].value_counts().reindex(DISTRICTS)
assert (counts.to_numpy() == counts_vc.to_numpy()).all()
assert counts.sum() == n_final
counts_nt = req_nt.groupby("district").size().reindex(DISTRICTS)
assert counts_nt.sum() == n_no_trash and n_no_trash + n_trash == n_final

rates = (pd.DataFrame({"district": DISTRICTS,
                       "request_count": counts.to_numpy(),
                       "request_count_no_trash": counts_nt.to_numpy()})
         .merge(demo[["district", POP_ESRI, POP_ACS]], on="district", validate="one_to_one")
         .rename(columns={POP_ESRI: "esri_population_2026", POP_ACS: "acs_population_2020_24"}))

RATE_COLS = {}
for suffix, count_col in [("", "request_count"), ("_no_trash", "request_count_no_trash")]:
    for denom, pop_col in [("esri", "esri_population_2026"), ("acs", "acs_population_2020_24")]:
        col = f"requests_per_1000_per_year_{denom}{suffix}"
        rates[col] = rate_per_1000(rates[count_col], rates[pop_col])
        RATE_COLS[f"{denom}{suffix or '_all'}"] = col

for key, col in RATE_COLS.items():
    rates[f"rank_{key.replace('_all', '')}"] = rates[col].rank(ascending=False, method="min").astype(int)
rates["ratio_to_lowest_esri"] = (rates["requests_per_1000_per_year_esri"]
                                 / rates["requests_per_1000_per_year_esri"].min())
assert (rates[list(RATE_COLS.values())] > 0).all().all()

rates = rates[["district", "request_count", "esri_population_2026", "acs_population_2020_24",
               "requests_per_1000_per_year_esri", "requests_per_1000_per_year_acs",
               "request_count_no_trash", "requests_per_1000_per_year_esri_no_trash",
               "requests_per_1000_per_year_acs_no_trash",
               "rank_esri", "rank_acs", "rank_esri_no_trash", "rank_acs_no_trash", "ratio_to_lowest_esri"]]
(rates.round({**{c: 1 for c in RATE_COLS.values()}, "ratio_to_lowest_esri": 2})
      .to_csv(OUT / "request_rates.csv", index=False))
rates.round(2)

### 2b. Chi-square goodness of fit → `chi_square_summary.csv`, `chi_square_district_diagnostics_esri.csv`

Expected count = total requests x the district's share of the population. `citywide_rate_per_1000_per_year` is the scenario's total requests over its total population: the rate every district would have if counts matched population shares, and the reference line on the bar chart.

In [ ]:
def gof(obs, pop):
    obs = np.asarray(obs, dtype=float)
    pop = np.asarray(pop, dtype=float)
    exp = obs.sum() * pop / pop.sum()
    chi2, p = stats.chisquare(obs, f_exp=exp)
    assert np.isclose(chi2, ((obs - exp) ** 2 / exp).sum())
    return chi2, p, exp


def w_size(w):
    return "negligible" if w < 0.1 else "small" if w < 0.3 else "medium" if w < 0.5 else "large"


rows = []
for req_set, count_col in [("all", "request_count"), ("no_trash", "request_count_no_trash")]:
    for denom, pop_col in [("esri", "esri_population_2026"), ("acs", "acs_population_2020_24")]:
        chi2, p, _ = gof(rates[count_col], rates[pop_col])
        n = int(rates[count_col].sum())
        w = float(np.sqrt(chi2 / n))
        rows.append({"scenario": f"{req_set}_{denom}", "request_set": req_set, "denominator": denom,
                     "n": n, "total_population": int(rates[pop_col].sum()),
                     "citywide_rate_per_1000_per_year": round(rate_per_1000(n, rates[pop_col].sum()), 1),
                     "chi_square": round(chi2, 1), "df": len(DISTRICTS) - 1, "p_value": fmt_p(p),
                     "cohens_w": round(w, 3), "w_size": w_size(w)})
chi_summary = pd.DataFrame(rows)
chi_summary.to_csv(OUT / "chi_square_summary.csv", index=False)
chi_summary

In [ ]:
_, _, exp_esri = gof(rates["request_count"], rates["esri_population_2026"])
obs = rates["request_count"].to_numpy()
diag = pd.DataFrame({
    "district": rates["district"],
    "observed": obs,
    "expected": exp_esri.round(1),
    "standardized_residual": ((obs - exp_esri) / np.sqrt(exp_esri)).round(2),
    "pct_above_below_expected": ((obs / exp_esri - 1) * 100).round(1),
    "population_share_pct": (rates["esri_population_2026"] / rates["esri_population_2026"].sum() * 100).round(1),
    "request_share_pct": (obs / obs.sum() * 100).round(1),
})
assert np.isclose(diag["expected"].sum(), n_final, atol=1)
diag.to_csv(OUT / "chi_square_district_diagnostics_esri.csv", index=False)
diag

### 2c. District covariates → `district_covariates.csv`

- **Renter share** = Esri 2026 renter-occupied housing units / total households x 100 (owner + renter = households in every district, checked in section 1).
- **Rent burden 30%+** = the sum of the four ACS bands from 30% of income up (30-34.9, 35-39.9, 40-49.9 and 50+). Its base is all renter-occupied units, including those where the share cannot be computed (the bands sum to about 100 with that group). The bands are published to one decimal, so the sum is rounded to one decimal; this keeps exact ties as ties instead of floating-point near-ties.
- **Spanish, English "not well"**: `acs_lang_18_64_speak_spanish_speak_english_not_well_pct` as published. Its denominator is not confirmed `[uncertain]` (`docs/ANALYSIS_STANDARDS.md` G-lang), and it leaves out the separate "not at all" row.
- Income and renter share are Esri 2026 estimates; internet, language and rent burden are ACS 2020-24 estimates with margins of error.

In [ ]:
RB_30PLUS = ["acs_rent_burden_30_34_9_pct_of_income_pct", "acs_rent_burden_35_39_9_pct_of_income_pct",
             "acs_rent_burden_40_49_9_pct_of_income_pct", "acs_rent_burden_50plus_pct_of_income_pct"]
COVARIATES = ["dip_hhinc_median_household_income_2026", "renter_share_2026_pct",
              "acs_internet_with_no_internet_access_pct",
              "acs_lang_18_64_speak_spanish_speak_english_not_well_pct", "rent_burden_30plus_pct"]

cov = demo[["district", "dip_hhinc_median_household_income_2026", "acs_internet_with_no_internet_access_pct",
            "acs_lang_18_64_speak_spanish_speak_english_not_well_pct"]].copy()
cov["renter_share_2026_pct"] = (demo["dip_summary_renter_occupied_housing_units_2026"]
                                / demo["dip_summary_total_households_2026"] * 100).round(1)
cov["rent_burden_30plus_pct"] = demo[RB_30PLUS].sum(axis=1).round(1)
cov = (rates[["district", "requests_per_1000_per_year_esri", "requests_per_1000_per_year_acs"]]
       .merge(cov, on="district", validate="one_to_one"))
cov = cov[["district", "requests_per_1000_per_year_esri", "requests_per_1000_per_year_acs", *COVARIATES]]
cov.round({"requests_per_1000_per_year_esri": 1, "requests_per_1000_per_year_acs": 1}).to_csv(
    OUT / "district_covariates.csv", index=False)
cov.round(1)

### 2d. Spearman rho → `spearman_rho.csv`

One row per covariate; one column per rate version (Esri or ACS denominator, all requests or Trash/Recycle removed). n = 6 districts: rho is descriptive only, with no p-value.

In [ ]:
RHO_COLS = {"rho_esri_all": RATE_COLS["esri_all"], "rho_acs_all": RATE_COLS["acs_all"],
            "rho_esri_no_trash": RATE_COLS["esri_no_trash"], "rho_acs_no_trash": RATE_COLS["acs_no_trash"]}
rho_in = cov[["district", *COVARIATES]].merge(rates[["district", *RHO_COLS.values()]], on="district",
                                              validate="one_to_one")
rho = pd.DataFrame([{"covariate": c, "n": len(rho_in),
                     **{k: round(float(stats.spearmanr(rho_in[c], rho_in[v]).statistic), 3)
                        for k, v in RHO_COLS.items()}}
                    for c in COVARIATES])
rho.to_csv(OUT / "spearman_rho.csv", index=False)
rho

### 2e. Filter log → `filter_log.csv`

In [ ]:
filter_log = pd.DataFrame({
    "measure": ["n_raw", "excluded_outside_window", "n_window", "excluded_no_district", "n_final",
                "n_trash_recycle", "n_no_trash", "trash_recycle_share_of_final_pct"],
    "value": pd.Series([n_raw, n_raw - n_window, n_window, n_window - n_final, n_final,
                        n_trash, n_no_trash, round(n_trash / n_final * 100, 1)], dtype=object),
    "description": ["rows in glendaleone_clean.csv",
                    "request_date outside 2020-01-01 to 2026-07-31",
                    "rows with request_date 2020-01-01 to 2026-07-31",
                    "rows in the window with no district (were N/A or NONE)",
                    "rows in the window in one of the 6 districts",
                    "n_final rows with request_type_group Trash/Recycle Services",
                    "n_final rows without Trash/Recycle Services",
                    "n_trash_recycle / n_final x 100"],
})
assert n_raw - (n_raw - n_window) - (n_window - n_final) == n_final
filter_log.to_csv(OUT / "filter_log.csv", index=False)
filter_log

## 3. Charts

Chart standard: `docs/ANALYSIS_STANDARDS.md` section 7. Titles are built from the numbers above, so they change if the data change; reread each one before it reaches a slide.

### 3a. Request rate by district (Esri 2026) → `request_rate_by_district_esri.png`

One hue (`BLUE`), since district is on the axis. Bars start at zero; values are labelled only on the highest and lowest bars and on the citywide reference line.

In [ ]:
Y = "requests_per_1000_per_year_esri"
bar = (rates.assign(label=rates["district"].map(nice))
            .sort_values(Y, ascending=False).reset_index(drop=True))
hi, lo = bar.iloc[0], bar.iloc[-1]
city = float(chi_summary.set_index("scenario").loc["all_esri", "citywide_rate_per_1000_per_year"])
title = f"{hi['label']} files {hi[Y] / lo[Y]:.2f}x the requests per resident of {lo['label']}"
subtitle = (f"GlendaleOne requests per 1,000 residents per year, {WINDOW_TXT}\n"
            f"Denominator: Esri 2026 population. n = {n_final:,} requests in the six districts")

fig, ax = plt.subplots(figsize=(12, 6.75))
fig.subplots_adjust(top=0.83, bottom=0.12, left=0.09, right=0.98)
sns.barplot(data=bar, x="label", y=Y, color=BLUE, ax=ax)
ax.axhline(city, color="#333333", lw=1.8, ls="--", zorder=3)
ax.text(len(bar) - 0.6, city + 1.2, f"Citywide {city:.1f}", ha="right", va="bottom",
        fontsize=15, color="#333333", bbox=dict(facecolor="white", edgecolor="none", pad=1.5), zorder=4)
for i in (0, len(bar) - 1):
    v = bar.loc[i, Y]
    ax.text(i, v + 1.2, f"{v:.1f}", ha="center", va="bottom", fontsize=17, fontweight="bold")
ax.set_ylim(0, bar[Y].max() * 1.15)
ax.set_xlabel("")
ax.set_ylabel("Requests per 1,000 residents per year", fontsize=15)
ax.set_title(subtitle, loc="left", fontsize=13, color="#444444", pad=12, linespacing=1.4)
fig.suptitle(title, x=ax.get_position().x0, y=0.985, ha="left", va="top", fontsize=22, fontweight="bold")
fig.text(ax.get_position().x0, 0.0, SOURCE, ha="left", va="top", fontsize=11, color="#555555")
fig.savefig(OUT / "request_rate_by_district_esri.png", dpi=200, bbox_inches="tight")
plt.show()

### 3b. Rate against each covariate, all requests vs Trash/Recycle removed → `scatter_panels_covariates.png`

One panel per covariate with a shared y-axis. Each district appears twice: filled for all requests and open for the rate with Trash/Recycle Services removed, joined by a grey line. Panel titles state each panel's finding with rho (Esri rates). The words are generated from rho: "near zero" when |rho| is under 0.2, "holds" when it keeps its sign at 0.2 or more, "reverses" when it flips sign at 0.2 or more. The sixth slot holds the key.

In [ ]:
PANELS = {
    "dip_hhinc_median_household_income_2026": ("Higher income", "income",
                                               "Median household income, Esri 2026"),
    "renter_share_2026_pct": ("More renters", "renters", "Renter share of households, Esri 2026"),
    "acs_internet_with_no_internet_access_pct": ("More homes offline", "internet access",
                                                 "No internet access, ACS 2020-24"),
    "acs_lang_18_64_speak_spanish_speak_english_not_well_pct": ("More limited English", "language",
                                                                "Age 18-64, Spanish, English not well, ACS"),
    "rent_burden_30plus_pct": ("More rent burden", "rent burden", "Renters paying 30%+ of income, ACS 2020-24"),
}
rho_e = rho.set_index("covariate")


def panel_title(lead, r_all, r_nt):
    if abs(r_all) < 0.2:
        first = f"{lead}: little link to requests (rho {r_all:+.2f})"
    else:
        first = f"{lead}: {'more' if r_all > 0 else 'fewer'} requests (rho {r_all:+.2f})"
    if abs(r_nt) < 0.2:
        second = f"Without Trash/Recycle: near zero ({r_nt:+.2f})"
    elif np.sign(r_nt) == np.sign(r_all):
        second = f"Without Trash/Recycle: holds ({r_nt:+.2f})"
    else:
        second = f"Without Trash/Recycle: reverses ({r_nt:+.2f})"
    return first + "\n" + second


def join_words(words):
    return words[0] if len(words) == 1 else ", ".join(words[:-1]) + " and " + words[-1]


fade = [c for c in COVARIATES
        if abs(rho_e.loc[c, "rho_esri_all"]) >= 0.4 and abs(rho_e.loc[c, "rho_esri_no_trash"]) < 0.2]
hold = [c for c in COVARIATES
        if abs(rho_e.loc[c, "rho_esri_no_trash"]) >= 0.5
        and np.sign(rho_e.loc[c, "rho_esri_no_trash"]) == np.sign(rho_e.loc[c, "rho_esri_all"])]
assert fade and hold
big_title = (f"Without Trash/Recycle, the links to {join_words([PANELS[c][1] for c in fade])} fade to near zero; "
             f"only {join_words([PANELS[c][1] for c in hold])} holds "
             f"(rho {rho_e.loc[hold[0], 'rho_esri_no_trash']:+.2f})")
big_sub = ("Requests per 1,000 residents per year by district, Esri 2026 population, "
           f"{WINDOW_TXT}. Filled: all requests. Open: Trash/Recycle Services removed. "
           "rho = Spearman, n = 6 districts, descriptive")

sc = cov[["district", *COVARIATES]].merge(
    rates[["district", RATE_COLS["esri_all"], RATE_COLS["esri_no_trash"]]], on="district")
sc["label"] = sc["district"].map(nice)
Y_ALL, Y_NT = RATE_COLS["esri_all"], RATE_COLS["esri_no_trash"]


def label_points(ax, xs, ys, names, avoid_x, avoid_y, fontsize=9):
    x0, x1 = ax.get_xlim()
    y0, y1 = ax.get_ylim()
    bb = ax.get_window_extent()
    w_pt, h_pt = bb.width / ax.figure.dpi * 72, bb.height / ax.figure.dpi * 72
    fx = (np.asarray(xs) - x0) / (x1 - x0)
    fy = (np.asarray(ys) - y0) / (y1 - y0)
    ax_x = (np.asarray(avoid_x) - x0) / (x1 - x0)
    ax_y = (np.asarray(avoid_y) - y0) / (y1 - y0)
    gx, gy, r_x, r_y = 6 / w_pt, 5 / h_pt, 5 / w_pt, 5 / h_pt
    placed = []

    def overlap(a, b):
        return max(0, min(a[1], b[1]) - max(a[0], b[0])) * max(0, min(a[3], b[3]) - max(a[2], b[2]))

    for i in np.argsort(fx):
        w = len(names[i]) * 0.58 * fontsize / w_pt
        h = 1.25 * fontsize / h_pt
        px, py = fx[i], fy[i]
        options = [
            ((px + gx, px + gx + w, py - h / 2, py + h / 2), px + gx, py, "left", "center"),
            ((px - gx - w, px - gx, py - h / 2, py + h / 2), px - gx, py, "right", "center"),
            ((px + 0.6 * gx, px + 0.6 * gx + w, py + gy, py + gy + h), px + 0.6 * gx, py + gy, "left", "bottom"),
            ((px + 0.6 * gx, px + 0.6 * gx + w, py - gy - h, py - gy), px + 0.6 * gx, py - gy, "left", "top"),
            ((px - 0.6 * gx - w, px - 0.6 * gx, py + gy, py + gy + h), px - 0.6 * gx, py + gy, "right", "bottom"),
            ((px - 0.6 * gx - w, px - 0.6 * gx, py - gy - h, py - gy), px - 0.6 * gx, py - gy, "right", "top"),
        ]
        best, best_cost = None, None
        for box, tx, ty, ha, va in options:
            cost = sum(overlap(box, b) for b in placed) * 100
            cost += sum(overlap(box, (mx - r_x, mx + r_x, my - r_y, my + r_y))
                        for mx, my in zip(ax_x, ax_y)) * 100
            cost += max(0, box[1] - 1) + max(0, -box[0]) + max(0, box[3] - 1) + max(0, -box[2])
            if best_cost is None or cost < best_cost - 1e-12:
                best, best_cost = (box, tx, ty, ha, va), cost
        box, tx, ty, ha, va = best
        placed.append(box)
        ax.text(tx, ty, names[i], transform=ax.transAxes, ha=ha, va=va, fontsize=fontsize, color="#222222")


fig, axes = plt.subplots(2, 3, figsize=(12, 6.75), sharey=True)
fig.subplots_adjust(left=0.085, right=0.99, top=0.765, bottom=0.135, hspace=0.68, wspace=0.08)
for ax, c in zip(axes.flat, COVARIATES):
    lead, _, xlabel = PANELS[c]
    for _, r in sc.iterrows():
        ax.plot([r[c], r[c]], [r[Y_NT], r[Y_ALL]], color=LINE_GREY, lw=1.3, zorder=1)
    ax.scatter(sc[c], sc[Y_ALL], s=46, color=BLUE, zorder=3)
    ax.scatter(sc[c], sc[Y_NT], s=46, facecolors="white", edgecolors=GREY, linewidths=1.5, zorder=3)
    span = sc[c].max() - sc[c].min()
    ax.set_xlim(sc[c].min() - 0.1 * span, sc[c].max() + 0.1 * span)
    ax.set_ylim(15, 85)
    ax.grid(lw=0.7, color="#e3e6ea")
    ax.set_title(panel_title(lead, rho_e.loc[c, "rho_esri_all"], rho_e.loc[c, "rho_esri_no_trash"]),
                 loc="left", fontsize=10.5, fontweight="bold", pad=6, linespacing=1.3)
    ax.set_xlabel(xlabel, fontsize=9.5)
    ax.tick_params(labelsize=9)
    fmt = (FuncFormatter(lambda v, _: f"${v / 1000:.0f}k") if c.startswith("dip_hhinc")
           else FuncFormatter(lambda v, _: f"{v:.0f}%"))
    ax.xaxis.set_major_formatter(fmt)
    label_points(ax, sc[c].to_numpy(), sc[Y_ALL].to_numpy(), sc["label"].tolist(),
                 np.r_[sc[c].to_numpy(), sc[c].to_numpy()], np.r_[sc[Y_ALL].to_numpy(), sc[Y_NT].to_numpy()])
for ax in axes[:, 0]:
    ax.set_ylabel("Requests per 1,000\nresidents per year", fontsize=10)

key = axes.flat[5]
key.axis("off")
handles = [Line2D([], [], marker="o", ls="", markersize=8, color=BLUE, label="All requests"),
           Line2D([], [], marker="o", ls="", markersize=8, markerfacecolor="white", markeredgecolor=GREY,
                  markeredgewidth=1.5, label="Trash/Recycle Services removed"),
           Line2D([], [], color=LINE_GREY, lw=1.6, label="Same district")]
key.legend(handles=handles, loc="upper left", bbox_to_anchor=(0.04, 1.0), frameon=False, fontsize=10,
           title="Each district appears twice", title_fontsize=10.5, alignment="left")
ties = [f"{join_words(g['label'].tolist())} share {PANELS[c][1]} ({g[c].iloc[0]:g}{'%' if c.endswith('_pct') else ''}), so their points overlap."
        for c in COVARIATES for _, g in sc.groupby(c) if len(g) > 1]
key.text(0.06, 0.38, textwrap.fill(" ".join(["Names sit next to the all-requests point. Both series use the "
                                             "Esri 2026 population; panel titles give rho for each series.",
                                             *ties]), 44),
         transform=key.transAxes, fontsize=9.5, color="#444444", va="top", linespacing=1.3)

fig.text(0.01, 0.99, textwrap.fill(big_title, 98), ha="left", va="top", fontsize=15, fontweight="bold",
         linespacing=1.25)
fig.text(0.01, 0.895, textwrap.fill(big_sub, 150), ha="left", va="top", fontsize=10, color="#444444",
         linespacing=1.3)
fig.text(0.01, 0.0, SOURCE + "\nCovariates: Esri 2026 (income, renters); ACS 2020-24 5-year estimates "
         "(internet, language, rent burden).", ha="left", va="top", fontsize=9, color="#555555",
         linespacing=1.3)
fig.savefig(OUT / "scatter_panels_covariates.png", dpi=200, bbox_inches="tight")
plt.show()

## 4. `findings.md`

Written by hand after the run. Every number in it is a cell in one of the CSVs above.